# What Is Object Detection?

### Classification vs Detection

So far we have been doing **classification** and **recognition**.

Both answer:

> *What is this?*

Object detection answers:

> *What is this, and **where** is it?*

That one extra word changes everything about how the model works.

### Output Comparison

| Task | Model Output | Example |
| ---- | ------------ | ------- |
| Classification | Label only | `"dog"` |
| Face Recognition | Embedding (512 numbers) | `[0.12, -0.33, ...]` |
| Object Detection | Label + Bounding Box | `"dog", (100, 50, 300, 250)` |

A **bounding box** is just 4 numbers describing a rectangle:

```
(x1, y1) ─────────┐
   │              │
   │     dog 🐕   │
   │              │
   └───────── (x2, y2)
```

We have already been using bounding boxes — MTCNN returned them for every face it detected.

### The Core Problem — Variable Output

Classification always produces **one** output.

But an image could contain:

```
1 dog
3 cars
2 people
```

That's 6 objects — and a different image might have 1, or 20.

Neural networks prefer **fixed-size outputs**.

So the question becomes:

> How do you make a neural network handle a variable number of objects?

### First Attempt — Region-Based Detection (R-CNN, 2014)

The first approach was intuitive:

```
Image
↓
Find ~2000 regions that might contain objects
↓
Run classifier on each region separately
↓
Output label + box for each match
```

It worked — but it was extremely slow.

```
2000 regions × 30 fps = 60,000 classifications per second
```

Not usable in real-time.

### The YOLO Insight

YOLO (**Y**ou **O**nly **L**ook **O**nce) asked a different question:

> What if we looked at the entire image **once** and predicted everything simultaneously?

<table>
    <tr>
        <th>R-CNN</th>
        <th>YOLO</th>
    </tr>
    <tr>
        <td>
            Image <br>
            ↓ <br>
            Region 1 → CNN <br>
            Region 2 → CNN <br>
            Region 3 → CNN <br>
        </td>
        <td>
            Image <br>
            ↓ <br>
            One CNN pass <br>
            ↓ <br>
            All boxes + labels at once <br>
        </td>
    </tr>
    <tr>
        <td>Slow 🐢</td>
        <td>Fast 🐇</td>
    </tr>
</table>

### How YOLO Solves The Variable Output Problem

YOLO divides the image into a fixed **grid**:

```
+-------+-------+-------+
|       |       |       |
|  cell |  cell |  cell |
|       |       |       |
+-------+-------+-------+
|       |       |       |
|  cell |  cell |  cell |
|       |       |       |
+-------+-------+-------+
|       |       |       |
|  cell |  cell |  cell |
|       |       |       |
+-------+-------+-------+
```

Each cell is responsible for detecting objects whose **center falls inside it**.

```
+-------+-------+-------+
|       |  🐕   |       |
|       |   •   |       |  ← dog center here
|       |       |       |     this cell detects it
+-------+-------+-------+
```

Grid is always fixed → output size is always fixed → neural network is happy.

### What Each Cell Predicts

Every cell always outputs the same structure:

```
x, y        ←  center of box (relative to cell)
w, h        ←  width and height of box
confidence  ←  is there actually an object here?
class probs ←  dog? car? person? ...
```

| Value | Meaning |
| ----- | ------- |
| `x, y` | Where is the object center inside the cell |
| `w, h` | How big is the bounding box |
| `confidence` | How sure is the model an object exists here |
| `class probs` | Which class is most likely |

### What About Multiple Objects In One Cell?

Good question — what if two objects share the same cell?

```
+-------+-------+
| 🐕🐈 |       |
|  ••   |       |  ← two centers, one cell
|       |       |
+-------+-------+
```

Early YOLO (v1) would miss one of them. This was a known limitation.

The fix came in two steps:

**1. Smaller grid = less overlap chance**

A 13×13 or 19×19 grid makes it much harder for two object centers to land in the same cell.

**2. Anchor Boxes**

Each cell predicts multiple boxes of different shapes simultaneously:

```
┌──────────┐  ← tall anchor   (good for: people)
│  ┌────┐  │  ← wide anchor   (good for: cars)
│  │    │  │  ← square anchor (good for: faces, balls)
│  └────┘  │
└──────────┘
```

Each anchor specializes in a different shape. One cell can now detect multiple objects.

| YOLO Version | Limitation Fixed |
| ------------ | ---------------- |
| v1 (2015) | One object per cell only |
| v2 (2016) | Anchor boxes added |
| v5 / v8 | Smarter anchors, multi-scale detection |

### Multi-Scale Detection

Modern YOLO runs detection at **multiple grid sizes** simultaneously:

```
Large grid   (52×52) → finds small objects  (ball, face)
Medium grid  (26×26) → finds medium objects (dog, person)
Small grid   (13×13) → finds large objects  (car, truck)
```

This is similar to how your eyes work — you notice big objects immediately, then focus on smaller details.

### Confidence & NMS (**N**on-**M**aximum **S**upression)

YOLO runs on every cell — so multiple nearby cells might all claim to detect the same dog:

```
┌─────────────────┐
│  box1 (0.91) ─┐ │
│  box2 (0.88)─┐│ │  ← all detecting the same dog
│  box3 (0.45)┐││ │
│             🐕  │
└─────────────────┘
```

**NMS** cleans this up:

```
Keep the highest confidence box
↓
Remove all other boxes that overlap it heavily
↓
One clean box per object ✅
```

| Term | Meaning |
| ---- | ------- |
| Confidence score | How sure the model is about this detection |
| IoU (Intersection over Union) | How much two boxes overlap |
| NMS threshold | If overlap > this, remove the weaker box |

### IoU — Intersection over Union

This is how YOLO measures if two boxes are detecting the same object:

```
┌──────────────┐
│   Box A      │
│      ┌───────┼────┐
│      │ INTER │    │
└──────┼───────┘    │
       │    Box B   │
       └────────────┘

IoU = Intersection Area / Union Area
```

| IoU Value | Meaning |
| --------- | ------- |
| 1.0 | Perfect overlap — same box |
| 0.5+ | Probably same object |
| 0.0 | No overlap at all |

If IoU > threshold → one of the boxes gets removed by NMS.

### Full YOLO Pipeline

```
Image
↓
Divide into grid
↓
CNN processes entire image once
↓
Each cell predicts: boxes + confidence + class
↓
Filter low confidence predictions
↓
Apply NMS (remove duplicate boxes)
↓
Final detections: label + bounding box
```

### Summary

| Concept | What It Means |
| ------- | ------------- |
| Bounding Box | Rectangle around detected object (x1,y1,x2,y2) |
| Grid | Image divided into cells, each responsible for nearby objects |
| Anchor Boxes | Multiple box shapes per cell to handle different object sizes |
| Confidence | How sure the model is something exists in a cell |
| IoU | How much two boxes overlap |
| NMS | Removes duplicate detections, keeps best box |
| Multi-scale | Detection at different grid sizes to catch small and large objects |

Next, we will install YOLO and run it on an image to see all of this in action.